# Huấn luyện VocabMaster AI với Unsloth (Qwen2.5-3B-Instruct)
Kịch bản này sử dụng Google Colab (Free T4 GPU) để fine-tune mô hình AI cục bộ bằng LoRA.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!cp /content/drive/MyDrive/vocab_dataset.jsonl /content/

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-61c18us3/unsloth_475f67815d174fa28fddf69825fd2ec4
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-61c18us3/unsloth_475f67815d174fa28fddf69825fd2ec4
  Resolved https://github.com/unslothai/unsloth.git to commit ef97f3c961acf7fe724f95edc61c16c00f5da39c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Khuyến nghị cho dữ liệu ngắn
dtype = None # Auto detect
load_in_4bit = True # Giảm VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",    # Tối ưu hóa memory
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.7.5 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [4]:
from datasets import load_dataset
# Hãy upload file vocab_dataset.jsonl lên Colab trước khi chạy block này
dataset = load_dataset("json", data_files="vocab_dataset.jsonl", split="train")

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "chatml")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass

dataset = dataset.map(formatting_prompts_func, batched = True,)


Generating train split: 0 examples [00:00, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_n9526veh/tokenizer_config.json.


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [6]:
from trl import SFTTrainer, SFTConfig # Đổi import ở dòng này
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = SFTConfig( # Sử dụng SFTConfig thay vì TrainingArguments
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer.train()

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
1,0.005640
2,0.005640
3,0.012072
4,0.009126
5,0.013761
6,0.006571
7,0.008349
8,0.007577
9,0.006127
10,0.006103


TrainOutput(global_step=60, training_loss=0.005390746231811742, metrics={'train_runtime': 116.2375, 'train_samples_per_second': 4.129, 'train_steps_per_second': 0.516, 'total_flos': 1224759595991040.0, 'train_loss': 0.005390746231811742, 'epoch': 60.0})

In [ ]:
# Lưu model và Export sang GGUF (4-bit quant)
model.save_pretrained_gguf("vocab_ai_gguf", tokenizer, quantization_method = "q4_k_m")
# Sau khi chạy xong, vào thư mục vocab_ai_gguf tải file .gguf về máy cá nhân.